# Target Encoding

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Create sample dataset
np.random.seed(42)
n_samples = 2000
cities = [f'City_{i}' for i in range(50)]
data = pd.DataFrame({
    'city': np.random.choice(cities, n_samples, p=np.random.dirichlet(np.ones(50)*0.5)),
    'age': np.random.randint(18, 70, n_samples),
    'income': np.random.randint(30000, 150000, n_samples)
})

# Create a target correlated with city
city_means = {city: np.random.uniform(0.1, 0.9) for city in cities}
data['converted'] = data['city'].apply(lambda c: np.random.binomial(1, p=city_means[c]))

X = data.drop('converted', axis=1)
y = data['converted']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
train_df = pd.concat([X_train, y_train], axis=1)

print("Target Encoding Example")
print("="*60)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Target Encoding Example
Train shape: (1500, 3), Test shape: (500, 3)


### Method 1: Naive Target Encoding (DANGEROUS - FOR DEMONSTRATION ONLY)

In [3]:

print("\n" + "="*60)
print("Method 1: Naive Target Encoding (Overfits!)")
print("="*60)

# Calculate means on the entire training set
city_target_mean = train_df.groupby('city')['converted'].mean()

# Apply to train and test
X_train_naive = X_train.copy()
X_test_naive = X_test.copy()

X_train_naive['city_encoded'] = X_train_naive['city'].map(city_target_mean)
X_test_naive['city_encoded'] = X_test_naive['city'].map(city_target_mean)

# Handle unseen categories in test set
global_mean = y_train.mean()
X_test_naive['city_encoded'].fillna(global_mean, inplace=True)

# Train model
model_naive = RandomForestClassifier(n_estimators=100, random_state=42)
model_naive.fit(X_train_naive[['city_encoded', 'age', 'income']], y_train)

train_auc = roc_auc_score(y_train, model_naive.predict_proba(X_train_naive[['city_encoded', 'age', 'income']])[:, 1])
test_auc = roc_auc_score(y_test, model_naive.predict_proba(X_test_naive[['city_encoded', 'age', 'income']])[:, 1])

print(f"Train AUC (Naive): {train_auc:.4f}")
print(f"Test AUC (Naive):  {test_auc:.4f}")
print(f"Overfitting Gap:   {train_auc - test_auc:.4f}")
print("⚠️ High training AUC and large gap indicate severe overfitting due to data leakage.")


Method 1: Naive Target Encoding (Overfits!)
Train AUC (Naive): 1.0000
Test AUC (Naive):  0.7396
Overfitting Gap:   0.2604
⚠️ High training AUC and large gap indicate severe overfitting due to data leakage.


### Method 2: Target Encoding with Cross-Validation (Robust Method)

In [5]:
print("\n" + "="*60)
print("Method 2: Target Encoding with K-Fold Cross-Validation")
print("="*60)

def target_encode_cv(train_series, target, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=False)
    encoded_series = pd.Series(index=train_series.index, dtype=float)
    
    for train_idx, val_idx in kf.split(train_series):
        # Calculate means on the training part of the fold
        means = target.iloc[train_idx].groupby(train_series.iloc[train_idx]).mean()
        
        # Apply to the validation part of the fold
        encoded_series.iloc[val_idx] = train_series.iloc[val_idx].map(means)
        
    # Fill any NaNs with global mean (for categories that only appear in one fold)
    encoded_series.fillna(target.mean(), inplace=True)
    
    return encoded_series

X_train_cv = X_train.copy()
X_test_cv = X_test.copy()

# Create leak-free encoding for training set
X_train_cv['city_encoded'] = target_encode_cv(X_train['city'], y_train)

# For test set, use means from the entire training set
full_train_means = y_train.groupby(X_train['city']).mean()
X_test_cv['city_encoded'] = X_test['city'].map(full_train_means).fillna(y_train.mean())

# Train model
model_cv = RandomForestClassifier(n_estimators=100, random_state=42)
model_cv.fit(X_train_cv[['city_encoded', 'age', 'income']], y_train)

train_auc_cv = roc_auc_score(y_train, model_cv.predict_proba(X_train_cv[['city_encoded', 'age', 'income']])[:, 1])
test_auc_cv = roc_auc_score(y_test, model_cv.predict_proba(X_test_cv[['city_encoded', 'age', 'income']])[:, 1])

print(f"Train AUC (CV): {train_auc_cv:.4f}")
print(f"Test AUC (CV):  {test_auc_cv:.4f}")
print(f"Overfitting Gap: {train_auc_cv - test_auc_cv:.4f}")
print("✅ CV-based encoding is the most robust way to prevent leakage.")


Method 2: Target Encoding with K-Fold Cross-Validation
Train AUC (CV): 1.0000
Test AUC (CV):  0.7474
Overfitting Gap: 0.2526
✅ CV-based encoding is the most robust way to prevent leakage.


### Method 3: Using category_encoders library

In [6]:

print("\n" + "="*60)
print("Method 3: Using category_encoders Library")
print("="*60)

try:
    from category_encoders import TargetEncoder
    
    # The library handles smoothing and CV internally
    encoder = TargetEncoder(cols=['city'], smoothing=30)
    
    X_train_lib = X_train.copy()
    X_test_lib = X_test.copy()
    
    # fit_transform on train, transform on test
    X_train_lib['city'] = encoder.fit_transform(X_train_lib['city'], y_train)
    X_test_lib['city'] = encoder.transform(X_test_lib['city'])
    
    model_lib = RandomForestClassifier(n_estimators=100, random_state=42)
    model_lib.fit(X_train_lib[['city', 'age', 'income']], y_train)
    
    train_auc_lib = roc_auc_score(y_train, model_lib.predict_proba(X_train_lib[['city', 'age', 'income']])[:, 1])
    test_auc_lib = roc_auc_score(y_test, model_lib.predict_proba(X_test_lib[['city', 'age', 'income']])[:, 1])
    
    print(f"Train AUC (Library): {train_auc_lib:.4f}")
    print(f"Test AUC (Library):  {test_auc_lib:.4f}")
    print("✅ The category_encoders library provides a production-ready, safe implementation.")

except ImportError:
    print("\n⚠️ Install category_encoders: pip install category-encoders")


Method 3: Using category_encoders Library
Train AUC (Library): 1.0000
Test AUC (Library):  0.7466
✅ The category_encoders library provides a production-ready, safe implementation.
